# Actividad: Inferencia lógica en las Minas de Moria

En esta actividad construiremos un mecanismo de inferencia basado en **lógica proposicional**, utilizando el algoritmo de **verificación de modelos mediante tablas de verdad** estudiado en clase.

El objetivo es implementar un procedimiento que permita determinar si una proposición $\alpha$ es consecuencia lógica de una base de conocimiento $BC$:

$$
BC \models \alpha
$$

Recordemos que esta relación se cumple cuando $\alpha$ es verdadera en **todos los modelos que satisfacen la base de conocimiento**.

## Contexto: Las Minas de Moria

Consideraremos un agente que se desplaza en un tablero de $4\times4$ casillas. En este entorno pueden existir pozos y un Troll, cuya ubicación es desconocida para el agente.

Aunque el agente no puede observar directamente estos peligros, recibe dos tipos de percepciones:

- **Brisa:** indica que existe al menos un pozo en alguna casilla adyacente.
- **Hedor:** indica que el Troll se encuentra en alguna casilla adyacente.

Estas percepciones pueden representarse mediante proposiciones lógicas:

- $P_{i,j}$: existe un pozo en la casilla $(i,j)$.
- $T_{i,j}$: el Troll se encuentra en la casilla $(i,j)$.
- $B_{i,j}$: se percibe brisa en la casilla $(i,j)$.
- $H_{i,j}$: se percibe hedor en la casilla $(i,j)$.

El agente utilizará las percepciones y las reglas del entorno para construir una **base de conocimiento (BC)** y razonar sobre la seguridad de las casillas.

Una casilla $(i,j)$ se considera segura cuando la base de conocimiento permite demostrar que no contiene un pozo ni al Troll:

$$
BC \models (\neg P_{i,j}\land\neg T_{i,j})
$$

## Objetivo de la actividad

Implementar el algoritmo de inferencia por tablas de verdad y utilizarlo para determinar si los movimientos de un agente a lo largo de una trayectoria predefinida son seguros.

Para ello, trabajaremos en tres bloques:

1. **Representación de la base de conocimiento:** representaremos sentencias lógicas mediante funciones de Python y construiremos una BC que pueda actualizarse incrementalmente.

2. **Implementación del algoritmo de inferencia:** implementaremos el algoritmo recursivo de verificación de modelos presentado en clase para comprobar si $BC\models\alpha$.

3. **Verificación de una trayectoria:** utilizaremos una matriz de percepciones y una trayectoria predefinida. En cada paso, el agente actualizará su conocimiento y determinará si puede demostrar que la siguiente casilla es segura.

**Importante:** el agente no conocerá directamente la ubicación de los peligros. Sus decisiones deberán basarse únicamente en las percepciones y las inferencias obtenidas a partir de su base de conocimiento.


# Bloque 1 — Representación y construcción de la base de conocimiento

En lógica proposicional, una **base de conocimiento (BC)** está formada por un conjunto de sentencias que representan hechos y reglas acerca del entorno.

Como vimos en clase, la base de conocimiento puede expresarse como la conjunción de todas las sentencias que contiene:

$$
BC=R_1\land R_2\land \cdots\land R_n
$$

donde cada $R_i$ representa una sentencia lógica.

Para implementar este concepto en Python, representaremos cada sentencia mediante una función que recibe un **modelo** y devuelve un valor booleano (`True` o `False`).


## 1. Representación de modelos

Un modelo es una asignación de valores de verdad a las proposiciones atómicas.

En Python podemos representar un modelo mediante un diccionario, donde las claves corresponden a los símbolos proposicionales y los valores indican si cada proposición es verdadera o falsa.

Por ejemplo, consideremos las proposiciones:

$$
P_{1,1},\quad P_{1,2},\quad P_{2,1},\quad B_{1,1}
$$

El siguiente diccionario representa un modelo posible:

In [1]:
modelo = {
    'P_1_1': False,
    'P_1_2': False,
    'P_2_1': True,
    'B_1_1': True
}

En este modelo no hay pozos en $(1,1)$ ni en $(1,2)$, pero existe uno en $(2,1)$ y se percibe brisa en $(1,1)$.

## 2. Representación de sentencias lógicas

Una sentencia lógica puede representarse mediante una función `lambda` que evalúa su valor de verdad en un modelo.

Por ejemplo, la sentencia:

$$
\neg P_{1,1}
$$

puede expresarse como:


In [2]:
sentencia = lambda m: not m['P_1_1']

Al evaluar esta función, obtenemos el valor de verdad de la sentencia en el modelo proporcionado:

In [3]:
sentencia(modelo)

True

De manera similar, una conjunción:

$$
\neg P_{1,1}\land B_{1,1}
$$

puede representarse como:

In [4]:
sentencia = lambda m: (
    not m['P_1_1'] and m['B_1_1']
)

## 3. Construcción incremental de la base de conocimiento

En lugar de escribir todas las sentencias de una base de conocimiento desde el inicio, podemos construirla incrementalmente.

Partiremos de una base de conocimiento vacía, representada mediante:

In [5]:
BC = lambda m: True

Esta función representa una base de conocimiento sin restricciones, por lo que resulta verdadera para cualquier modelo.

Cuando el agente obtiene nueva información, podemos incorporarla mediante una conjunción:

$$
BC_{\text{nueva}}=BC_{\text{anterior}}\land R
$$

donde $R$ representa la nueva sentencia.

Para realizar esta operación utilizaremos la función `decir`, correspondiente a la operación **TELL** de un agente lógico:

In [6]:
def decir(BC, sentencia):
    return lambda m: BC(m) and sentencia(m)

La función recibe una base de conocimiento y una nueva sentencia, y devuelve una función que representa la conjunción de ambas.

De esta manera podemos agregar conocimiento progresivamente sin modificar las sentencias incorporadas anteriormente.

## 4. Ejemplo: conocimiento inicial del agente

Supongamos que el agente comienza en la casilla $(1,1)$, donde no existe un pozo y no percibe brisa.

Estas observaciones se representan mediante:

$$
\neg P_{1,1}
$$

$$
\neg B_{1,1}
$$

Podemos construir la base de conocimiento de la siguiente manera:

In [7]:
BC = lambda m: True

BC = decir(BC, lambda m: not m['P_1_1'])
BC = decir(BC, lambda m: not m['B_1_1'])

La base de conocimiento resultante representa:

$$
BC=\neg P_{1,1}\land\neg B_{1,1}
$$

Ahora evaluemos la base de conocimiento utilizando el modelo definido anteriormente:

In [8]:
print(BC(modelo))

False


El resultado será `False`, ya que el modelo establece que existe brisa en $(1,1)$, mientras que la base de conocimiento contiene la sentencia $\neg B_{1,1}$.

Esto significa que **el modelo no satisface la base de conocimiento**.

## Pregunta de reflexión

Si una base de conocimiento se evalúa como `False` en un determinado modelo, ¿significa que la base de conocimiento es incorrecta?

Explica tu respuesta considerando la diferencia entre una sentencia lógica y un modelo que satisface dicha sentencia.

**Respuesta**

No, el hecho de que una base de conocimiento ($BC$) se evalúe como `False` en un determinado modelo no significa que la $BC$ sea incorrecta ni contradictoria. Esto se debe a la diferencia fundamental entre el papel que desempeña cada elemento en la lógica proposicional: mientras que una sentencia lógica expresa un hecho o regla que restringe lo que sabemos que es real sobre el entorno, un modelo es simplemente una suposición o asignación hipotética de valores de verdad (`True` o `False`) que representa uno de los múltiples escenarios posibles del mundo.  

Dentro del proceso de razonamiento, los modelos plantean todas las combinaciones posibles que podrían existir, mientras que la base de conocimiento funciona como un filtro que delimita cuáles de esas suposiciones son compatibles con las percepciones y reglas del agente. Cuando un modelo satisface todas las sentencias de la $BC$ ($M \models BC$), la evaluación devuelve `True`, indicando que ese escenario es válido. En cambio, obtener `False` simplemente significa que esa suposición en particular contradice lo que el agente sabe y, por lo tanto, debe descartarse como un mundo imposible, sin que ello invalide la consistencia de la base de conocimiento.

Esto se observa en el ejemplo anterior, donde la $BC$ registra que no percibe brisa en la casilla (1,1) con $\neg B_{1,1}$, pero se evalúa frente a un modelo que supone sí hay brisa ($B_{1,1}$). Al existir una contradicción entre la suposición del modelo y la realidad registrada por el agente, el resultado es `False`, demostrando únicamente que dicho modelo no satisface la base de conocimiento. 

# Bloque 2 — Implementación del algoritmo de inferencia por tablas de verdad

En el bloque anterior representamos una **base de conocimiento (BC)** mediante una función de Python y utilizamos la operación `decir` para incorporar nuevas sentencias lógicas.

Ahora implementaremos el algoritmo de **inferencia por tablas de verdad** presentado en clase.

Recordemos que una proposición $\alpha$ es consecuencia lógica de una base de conocimiento cuando:

$$
BC \models \alpha
$$

Esto significa que $\alpha$ debe ser verdadera en **todos los modelos que satisfacen la base de conocimiento**.

Para comprobar esta relación, el algoritmo:

1. Identifica los símbolos proposicionales presentes en $BC$ y $\alpha$.
2. Genera todas las combinaciones posibles de valores de verdad para dichos símbolos.
3. Evalúa la base de conocimiento en cada modelo.
4. Si encuentra un modelo donde $BC$ es verdadera y $\alpha$ es falsa, devuelve `False`.
5. Si no existe ningún contraejemplo, devuelve `True`.

Para un conjunto de $n$ símbolos proposicionales existen $2^n$ modelos posibles.

## Pseudocódigo del algoritmo

El algoritmo utiliza dos funciones:

**¿Implicación-En-TV?(BC, $\alpha$)**

```text
función IMPLICACIÓN-EN-TV(BC, α)
    símbolos ← símbolos proposicionales de BC y α
    devolver COMPROBAR-TV(BC, α, símbolos, {})
```

**Comprobar-TV(BC, $\alpha$, símbolos, modelo)**

```text
función COMPROBAR-TV(BC, α, símbolos, modelo)

    si símbolos está vacío entonces
        si BC es verdadera en modelo entonces
            devolver α es verdadera en modelo
        si no
            devolver verdadero

    P ← primer símbolo
    resto ← símbolos restantes

    devolver
        COMPROBAR-TV(BC, α, resto, EXTENDER(P, verdadero, modelo))
        Y
        COMPROBAR-TV(BC, α, resto, EXTENDER(P, falso, modelo))
```

El algoritmo construye los modelos recursivamente, asignando a cada símbolo proposicional los valores `True` y `False`.

**Observación:** cuando la base de conocimiento es falsa en un modelo, el algoritmo devuelve `True` para esa rama porque dicho modelo no constituye un contraejemplo de la consecuencia lógica.


## Actividad 2.1 — Construcción incremental de modelos

Para generar las combinaciones de valores de verdad necesitamos construir modelos parciales.

Implementa la función:

`extender(p, valor, modelo)`

La función debe:

1. Recibir un símbolo proposicional `p`, un valor booleano `valor` y un diccionario `modelo`.
2. Crear una copia del modelo recibido.
3. Incorporar al nuevo modelo la asignación `p = valor`.
4. Devolver el nuevo modelo sin modificar el original.

Por ejemplo, al ejecutar:

```python
modelo = {'P_1_1': False}
nuevo = extender('B_1_1', True, modelo)
```

El resultado esperado para `nuevo` es:

```python
{'P_1_1': False, 'B_1_1': True}
```

In [9]:

# Carlos Roberto Vázquez Villegas
# Inteligencia Artificial 
# Grupo 1

# función extender
def extender(p, valor, modelo):
    nuevo_modelo = modelo.copy()
    nuevo_modelo[p] = valor
    return nuevo_modelo


In [10]:
# verificación
modelo = {'P_1_1': False}

nuevo = extender('B_1_1', True, modelo)

print("Modelo original:", modelo)
print("Modelo extendido:", nuevo)

Modelo original: {'P_1_1': False}
Modelo extendido: {'P_1_1': False, 'B_1_1': True}


## Actividad 2.2 — Implementación de `comprobar_tv`

A partir del pseudocódigo presentado anteriormente, implementa la función recursiva:

`comprobar_tv(BC, alfa, simbolos, modelo)`

Donde:

- `BC`: función que representa la base de conocimiento.
- `alfa`: función que representa la proposición que queremos verificar.
- `simbolos`: lista de símbolos proposicionales pendientes de asignar.
- `modelo`: diccionario que contiene las asignaciones de verdad realizadas.

La función debe considerar dos casos:

**Caso base**

Cuando no quedan símbolos por asignar, el modelo está completo.

- Si `BC(modelo)` es `True`, devuelve el resultado de `alfa(modelo)`.
- En caso contrario, devuelve `True`, porque el modelo no satisface la base de conocimiento.

**Caso recursivo**

Cuando todavía existen símbolos pendientes:

1. Selecciona el primer símbolo de la lista.
2. Construye dos modelos parciales, asignando `True` y `False` al símbolo seleccionado.
3. Realiza una llamada recursiva para cada asignación.
4. Combina los resultados utilizando el operador lógico `and`.

**Importante:** utiliza la función `extender` implementada anteriormente. No es necesario almacenar todos los modelos en una lista.


In [11]:
def comprobar_tv(BC, alfa, simbolos, modelo):

    # Caso base
    if len(simbolos) == 0:
        if BC(modelo):
            return alfa(modelo)
        else:
            return True
        
    # Caso recursivo
    p = simbolos[0]
    resto = simbolos[1:]

    return (
        comprobar_tv(BC, alfa, resto, extender(p, True, modelo))
        and
        comprobar_tv(BC, alfa, resto, extender(p, False, modelo))
    )

## Actividad 2.3 — Verificación de consecuencias lógicas

Utilizaremos el algoritmo implementado para comprobar inferencias sobre una base de conocimiento sencilla.

Consideremos las siguientes sentencias:

$$
R_1:\neg P_{1,1}
$$

$$
R_2:B_{1,1}\leftrightarrow(P_{1,2}\lor P_{2,1})
$$

$$
R_3:\neg B_{1,1}
$$

La base de conocimiento es:

$$
BC=R_1\land R_2\land R_3
$$

La segunda sentencia establece que existe brisa en $(1,1)$ si y solo si existe al menos un pozo en alguna de las dos casillas adyacentes.

La tercera sentencia indica que el agente no percibe brisa.

**Primero verificaremos:**

$$
BC\models\neg P_{1,2}
$$

**Después verificaremos:**

$$
BC\models\neg P_{2,1}
$$

Utiliza la función `comprobar_tv` para determinar si ambas proposiciones son consecuencias lógicas de la base de conocimiento.


In [12]:
# Base de conocimiento inicial
BC = lambda m: True

# R1: No existe un pozo en (1,1)
BC = decir(BC, lambda m: not m['P_1_1'])

# R2: B_1_1 <-> (P_1_2 or P_2_1)
BC = decir(BC, lambda m: (
    m['B_1_1'] ==
    (m['P_1_2'] or m['P_2_1'])
))

# R3: No se percibe brisa en (1,1)
BC = decir(BC, lambda m: not m['B_1_1'])

# Símbolos proposicionales
simbolos = ['P_1_1', 'P_1_2', 'P_2_1', 'B_1_1']

In [13]:
# Consulta 1: BC |= ¬P_1_2
alfa = lambda m: not m['P_1_2']

resultado = comprobar_tv(BC, alfa, simbolos, {})
print("BC implica ¬P_1_2:", resultado)

# Consulta 2: BC |= ¬P_2_1
alfa = lambda m: not m['P_2_1']

resultado = comprobar_tv(BC, alfa, simbolos, {})
print("BC implica ¬P_2_1:", resultado)


BC implica ¬P_1_2: True
BC implica ¬P_2_1: True


## Preguntas de reflexión

1. ¿Por qué la base de conocimiento permite concluir que no existen pozos en las casillas $(1,2)$ y $(2,1)$, aunque el agente no las haya visitado?

    **Respuesta**

    Esto es posible gracias a la sentencia $R_2$ ($B_{1,1} \leftrightarrow (P_{1,2} \lor P_{2,1})$), la cual establece que existe brisa en $(1,1)$ si y solo si existe al menos un pozo en alguna de las dos casillas adyacentes. Al combinarse con la sentencia $R_3$ ($\neg B_{1,1}$), que indica como un hecho que el agente no percibe brisa en su posición actual, los únicos modelos que satisfacen simultáneamente toda la base de conocimiento exigen que la disyunción $(P_{1,2} \lor P_{2,1})$ sea falsa. Para que dicha disyunción resulte falsa, ambos componentes deben ser necesariamente falsos ($\neg P_{1,2} \land \neg P_{2,1}$), permitiendo al agente deducir con certeza la ausencia de pozos en ambas casillas sin necesidad de haberlas visitado previamente.  

2. ¿Qué ocurriría si elimináramos de la base de conocimiento la sentencia $\neg B_{1,1}$? ¿Seguirían siendo válidas las dos inferencias?

    **Respuesta**

    Si se eliminara la sentencia $R_3$ ($\neg B_{1,1}$), la base de conocimiento quedaría conformada únicamente por $R_1$ ($\neg P_{1,1}$) y $R_2$ ($B_{1,1} \leftrightarrow (P_{1,2} \lor P_{2,1})$), perdiendo la información sobre la percepción real del agente en la casilla $(1,1)$. Por tanto, ninguna de las dos inferencias seguiría siendo válida (ambas devolverían `False`). Sin la restricción de que no hay brisa, existen modelos que satisfacen la base de conocimiento en los cuales $B_{1,1}$ es verdadera y existe un pozo en $(1,2)$, en $(2,1)$ o en ambas casillas.

3. ¿Por qué el algoritmo utiliza `and` para combinar los resultados de las dos llamadas recursivas?

    **Respuesta**

    Al dividir recursivamente cada símbolo proposicional en dos asignaciones (`True` y `False`), el algoritmo construye y recorre un árbol binario de modelos cuyas hojas representan todas las combinaciones posibles de valores de verdad. Por definición, demostrar que $BC \models \alpha$ equivale a comprobar que la expresión $BC \land \neg\alpha$ es insatisfacible; es decir, que en ninguna hoja del árbol puede existir un contraejemplo donde la base de conocimiento sea verdadera y $\alpha$ sea falsa. 
    
    Dado que una hoja devuelve `False` precisamente cuando encuentra uno de estos contraejemplos, combinar las dos llamadas recursivas mediante el operador lógico `and` garantiza que todas las ramas del árbol cumplan con la implicación lógica: basta con que una sola subrama encuentre un modelo que invalide $\alpha$ para que el `and` propague `False` como resultado final

4. Si una base de conocimiento contiene $n=10$ símbolos proposicionales, ¿cuántos modelos posibles existen?

    **Respuesta**

    Existen exactamente $1,024$ modelos posibles. Dado que cada símbolo proposicional puede tomar dos valores de verdad independientes (`True` o `False`), el número total de combinaciones posibles crece de forma exponencial según la fórmula $2^n$, lo que para $n = 10$ corresponde a $2^{10} = 1,024$ asignaciones distintas.


# Bloque 3 — Verificación de una trayectoria en las Minas de Moria

En el bloque anterior implementamos el algoritmo `comprobar_tv`, que permite determinar si una proposición es consecuencia lógica de una base de conocimiento:

$$
BC \models \alpha
$$

Ahora utilizaremos este algoritmo para evaluar los movimientos de un agente en las Minas de Moria.

El agente se encuentra en un tablero de $4\times4$ casillas y debe seguir una **trayectoria predefinida**. Sin embargo, antes de avanzar a cada casilla, debe demostrar que no contiene un pozo ni al Troll.

Una casilla $(i,j)$ se considera segura cuando:

$$
BC \models (\neg P_{i,j}\land\neg T_{i,j})
$$

## Reglas del entorno

El agente recibe dos percepciones:

- **Brisa:** indica que existe al menos un pozo en una casilla adyacente.
- **Hedor:** indica que el Troll se encuentra en alguna casilla adyacente.

Estas percepciones obedecen las siguientes reglas:

$$
B_{i,j}\leftrightarrow
\bigvee_{(x,y)\in Adj(i,j)}P_{x,y}
$$

$$
H_{i,j}\leftrightarrow
\bigvee_{(x,y)\in Adj(i,j)}T_{x,y}
$$

donde $Adj(i,j)$ representa el conjunto de casillas adyacentes a $(i,j)$, considerando movimientos horizontales y verticales.

## Procedimiento

Para verificar la trayectoria, el agente realizará las siguientes operaciones:

1. Consultar las percepciones de su posición actual.
2. Incorporar a la base de conocimiento las restricciones derivadas de esas percepciones.
3. Construir una proposición que represente la seguridad de la siguiente casilla.
4. Utilizar `comprobar_tv` para verificar si dicha proposición es consecuencia lógica de la BC.
5. Avanzar únicamente si puede demostrar que la siguiente casilla es segura.

El proceso termina cuando no puede justificar el siguiente movimiento o cuando alcanza el final de la trayectoria.

**Importante:** que una casilla no pueda demostrarse segura no significa necesariamente que contenga un peligro. Puede significar que la información disponible es insuficiente.


## 1. Escenario de exploración

Utilizaremos un tablero de $4\times4$ casillas. Cada posición tiene asociado un vector de percepciones:

$$
(brisa, hedor)
$$

donde `True` indica que la percepción está presente y `False` que está ausente.

Las percepciones se encuentran almacenadas en un diccionario, que representa las observaciones que el entorno puede proporcionar al agente.

El agente únicamente podrá consultar la percepción de la casilla donde se encuentra.

## Trayectoria propuesta

El agente inicia en $(1,1)$ y debe intentar recorrer la siguiente trayectoria:

$$
(1,1)\rightarrow(1,2)\rightarrow(2,2)\rightarrow(2,3)
$$

Aunque la trayectoria está definida, **el agente no podrá avanzar automáticamente**. Cada movimiento deberá justificarse mediante una inferencia lógica.

Supondremos que la casilla inicial $(1,1)$ es segura.

El objetivo es determinar hasta qué posición puede avanzar el agente utilizando únicamente el conocimiento obtenido durante su recorrido.


In [14]:
N = 4

In [15]:
# Percepciones del entorno: (brisa, hedor)
percepciones = {
    (1,1): (False, False),
    (1,2): (False, False),
    (1,3): (False, False),
    (1,4): (False, False),

    (2,1): (False, False),
    (2,2): (True,  False),
    (2,3): (False, False),
    (2,4): (False, False),

    (3,1): (True,  False),
    (3,2): (False, False),
    (3,3): (True,  False),
    (3,4): (False, True),

    (4,1): (False, False),
    (4,2): (True,  False),
    (4,3): (False, True),
    (4,4): (False, False)
}

In [16]:
# Trayectoria propuesta
trayectoria = [
    (1,1),
    (1,2),
    (2,2),
    (2,3)
]

In [17]:
def percibir(posicion):
    return percepciones[posicion]

## 2. Traducción de percepciones a sentencias lógicas

Para actualizar la base de conocimiento necesitamos traducir las percepciones del agente a expresiones de lógica proposicional.

Por ejemplo, si el agente se encuentra en $(1,1)$ y no percibe brisa:

$$
\neg B_{1,1}
$$

De acuerdo con las reglas del entorno:

$$
B_{1,1}\leftrightarrow(P_{1,2}\lor P_{2,1})
$$

Por lo tanto, podemos incorporar directamente a la BC la restricción:

$$
\neg(P_{1,2}\lor P_{2,1})
$$

De manera similar, la ausencia de hedor permite incorporar:

$$
\neg(T_{1,2}\lor T_{2,1})
$$

Para simplificar la implementación, construiremos estas restricciones directamente a partir de las percepciones. Esto evita introducir símbolos adicionales para brisa y hedor en la enumeración de modelos.

La siguiente función obtiene las casillas adyacentes válidas dentro del tablero.


In [18]:
def casillas_adyacentes(i, j, N):

    candidatos = [
        (i-1, j),
        (i+1, j),
        (i, j-1),
        (i, j+1)
    ]

    return [
        (x, y) for x, y in candidatos
        if 1 <= x <= N and 1 <= y <= N
    ]

## 3. Incorporación de las percepciones a la BC

Utilizaremos la función `regla_percepcion` para construir una sentencia lógica a partir de las percepciones del agente.

La función recibe:

- La posición $(i,j)$.
- Los valores booleanos de brisa y hedor.
- El tamaño del tablero $N$.

La sentencia generada será verdadera únicamente cuando la presencia o ausencia de peligros en las casillas adyacentes sea consistente con ambas percepciones.

Después, utilizaremos la función `decir` del bloque 1 para incorporar esta sentencia a la base de conocimiento.

También necesitaremos actualizar el conjunto de símbolos proposicionales considerados durante la verificación de modelos.

In [19]:
def regla_percepcion(i, j, brisa, hedor, N):

    adyacentes = casillas_adyacentes(i, j, N)

    return lambda m: (
        (any(m[f'P_{x}_{y}'] for x, y in adyacentes) == brisa)
        and
        (any(m[f'T_{x}_{y}'] for x, y in adyacentes) == hedor)
    )

In [20]:
def agregar_simbolos(simbolos, i, j, N):

    for x, y in casillas_adyacentes(i, j, N):
        simbolos.add(f'P_{x}_{y}')
        simbolos.add(f'T_{x}_{y}')

    return simbolos

## Actividad 3.1 — Verificación de una casilla segura

Antes de recorrer toda la trayectoria, implementaremos una función que determine si una casilla puede considerarse segura.

La función:

`es_segura(i, j, BC, simbolos)`

debe construir la proposición:

$$
\alpha=\neg P_{i,j}\land\neg T_{i,j}
$$

y verificar si:

$$
BC\models\alpha
$$

Para ello, utiliza la función `comprobar_tv` implementada en el bloque anterior.

La función deberá devolver:

- `True` si la base de conocimiento permite demostrar que la casilla es segura.
- `False` si no es posible demostrarlo.

Recuerda que `False` no necesariamente significa que la casilla contiene un peligro.

In [23]:
def es_segura(i, j, BC, simbolos):

    # Construye la proposición alfa
    alfa = lambda m: (
        not m[f'P_{i}_{j}'] and
        not m[f'T_{i}_{j}']
    )

    return comprobar_tv(BC, alfa, list(simbolos), {})

## Actividad 3.2 — Verificación de la trayectoria

Finalmente, utilizaremos las funciones desarrolladas para verificar la trayectoria propuesta.

En cada posición, el agente deberá:

1. Consultar su vector de percepciones.
2. Incorporar las nuevas restricciones a la BC mediante `decir`.
3. Actualizar el conjunto de símbolos proposicionales.
4. Verificar si la siguiente casilla de la trayectoria es segura.
5. Continuar únicamente cuando la inferencia sea válida.

La siguiente implementación proporciona el ciclo de verificación.

**Completa la instrucción que determina si el siguiente movimiento está lógicamente justificado.**

In [24]:
def verificar_trayectoria(trayectoria):

    BC = lambda m: True
    simbolos = set()

    # Conocimiento inicial: la casilla (1,1) es segura
    BC = decir(BC, lambda m: not m['P_1_1'])
    BC = decir(BC, lambda m: not m['T_1_1'])

    simbolos.update(['P_1_1', 'T_1_1'])

    for paso in range(len(trayectoria) - 1):

        actual = trayectoria[paso]
        siguiente = trayectoria[paso + 1]

        i, j = actual

        brisa, hedor = percibir(actual)

        # Incorporar la percepción a la BC
        sentencia = regla_percepcion(i, j, brisa, hedor, N)
        BC = decir(BC, sentencia)

        # Actualizar símbolos
        simbolos = agregar_simbolos(simbolos, i, j, N)

        # Completa la verificación del movimiento
        sig_i, sig_j = siguiente
        segura = es_segura(sig_i, sig_j, BC, simbolos)

        print("Posición actual:", actual)
        print("Percepción:", (brisa, hedor))
        print("Siguiente casilla:", siguiente)
        print("¿Es segura?:", segura)
        print()

        if not segura:
            print("Movimiento no autorizado.")
            break

    else:
        print("Trayectoria completada.")

In [25]:
verificar_trayectoria(trayectoria)

Posición actual: (1, 1)
Percepción: (False, False)
Siguiente casilla: (1, 2)
¿Es segura?: True

Posición actual: (1, 2)
Percepción: (False, False)
Siguiente casilla: (2, 2)
¿Es segura?: True

Posición actual: (2, 2)
Percepción: (True, False)
Siguiente casilla: (2, 3)
¿Es segura?: False

Movimiento no autorizado.


## Preguntas de reflexión

1. ¿Por qué el agente puede demostrar que las casillas $(1,2)$ y $(2,2)$ son seguras sin conocer directamente la ubicación de los peligros?

    **Respuesta**
    
    Esto se debe a que el agente utiliza sus percepciones locales y las reglas del entorno para descartar la presencia de peligros en sus casillas adyacentes mediante consecuencia lógica  ($BC \models (\neg P_{i,j} \land \neg T_{i,j})$). Al encontrarse en $(1,1)$ y percibir (False, False) (sin brisa ni hedor), la regla de percepción exige que ninguna de sus casillas vecinas —$(1,2)$ y $(2,1)$— contenga un pozo ni al Troll, permitiendo demostrar que $(1,2)$ es segura. Posteriormente, al avanzar a $(1,2)$ y percibir nuevamente (False, False), el mismo mecanismo descarta peligros en todas sus adyacentes: $(1,1)$, $(1,3)$ y $(2,2)$, autorizando el movimiento hacia $(2,2)$ sin necesidad de observar dónde están ubicados los peligros reales en el tablero.    

2. ¿Por qué el agente no puede autorizar el movimiento hacia $(2,3)$? ¿Esto significa que necesariamente existe un peligro en esa casilla?

    **Respuesta**

    Al llegar a $(2,2)$, el agente percibe brisa (True, False), lo cual incorpora a la base de conocimiento la restricción de que debe existir al menos un pozo en sus casillas adyacentes: $P_{1,2} \lor P_{3,2} \lor P_{2,1} \lor P_{2,3}$. Aunque la base de conocimiento acumulada ya permite descartar a $(1,2)$ y $(2,1)$ por percepciones anteriores, aún quedan dos casillas candidatas sin explorar: $(3,2)$ y $(2,3)$. Como existen modelos válidos donde el pozo está en $(3,2)$ y no en $(2,3)$, pero también modelos válidos donde el pozo sí está en $(2,3)$, este último caso actúa como contraejemplo y comprobar_tv devuelve False. 

    Esto no significa necesariamente que exista un pozo en $(2,3)$ (de hecho, las percepciones de $(1,3)$ y $(2,4)$ muestran que $(2,3)$ no tiene peligros); simplemente indica que, con la información recopilada hasta ese momento, el agente enfrenta incertidumbre y no puede garantizar lógicamente la seguridad de la casilla.

3. ¿Qué ventaja tiene conservar las percepciones anteriores en la base de conocimiento, en lugar de razonar únicamente a partir de la percepción actual?

    **Respuesta**

    Conservar las percepciones históricas permite acumular restricciones sobre el entorno (propiedad de monotonía de la lógica proposicional), reduciendo progresivamente el conjunto de modelos posibles que satisfacen la base de conocimiento. Si el agente razonara únicamente con la percepción actual en $(2,2)$, tendría que considerar cuatro casillas sospechosas de contener un pozo ($(1,2)$, $(3,2)$, $(2,1)$ y $(2,3)$). Al mantener en la $BC$ las percepciones previas de $(1,1)$ y $(1,2)$, el agente ya sabe que $(1,2)$ y $(2,1)$ son seguras, acotando la incertidumbre únicamente a $(3,2)$ y $(2,3)$ y permitiendo tomar decisiones más informadas conforme explora.

4. ¿Cómo afecta el número de símbolos proposicionales al costo computacional del algoritmo `comprobar_tv`?

    **Respuesta**

    El costo computacional en tiempo crece de manera exponencial ($O(2^n)$) respecto al número $n$ de símbolos proposicionales incluidos en la inferencia. Cada vez que el agente visita una nueva casilla e incorpora los símbolos de peligro ($P_{x,y}$ y $T_{x,y}$) de sus celdas adyacentes, cada símbolo adicional duplica el tamaño del árbol binario de búsqueda de modelos. En consecuencia, el tiempo de ejecución se multiplica por dos por cada nuevo símbolo agregado, volviendo el algoritmo rápidamente costoso a medida que se exploran más casillas.   

## Conclusión

En esta actividad implementamos un mecanismo de inferencia lógica que permite tomar decisiones a partir de una base de conocimiento.

El agente no necesita conocer completamente el entorno para justificar sus movimientos. Utiliza las percepciones disponibles, las incorpora a la BC y verifica si la seguridad de una casilla es una consecuencia lógica del conocimiento acumulado.

Este procedimiento corresponde al ciclo básico de un agente basado en conocimiento:

$$
\boxed{\text{Percibir}\rightarrow\text{Actualizar BC}\rightarrow
\text{Inferir}\rightarrow\text{Actuar}}
$$

# Reto final — Complejidad de la verificación de modelos

El algoritmo `comprobar_tv` verifica si una proposición es consecuencia lógica de una base de conocimiento mediante la enumeración de modelos.

Como estudiamos en clase, si existen $n$ símbolos proposicionales, el número de modelos posibles es:

$$
M(n)=2^n
$$

Esto significa que cada vez que se incorpora un nuevo símbolo proposicional, el número de modelos posibles se duplica.

## Actividad

Modifica la función `verificar_trayectoria` para que, en cada posición visitada, registre:

1. La posición actual del agente.
2. El número de símbolos proposicionales utilizados en la inferencia.
3. El número de modelos posibles correspondiente a esos símbolos.

Al finalizar la ejecución, muestra una tabla con la información recopilada.

Para calcular el número de modelos posibles, utiliza:

```python
modelos_posibles = 2 ** len(simbolos)
```

**Nota:** este cálculo proporciona el número de asignaciones posibles, no necesariamente el número de modelos que satisfacen la BC ni los efectivamente evaluados por el algoritmo, ya que Python puede interrumpir algunas ramas mediante la evaluación de cortocircuito.

## Preguntas de reflexión

1. ¿Qué ocurre con el número de modelos posibles cuando se agrega un símbolo proposicional?

    **Respuesta**
    
    Cada vez que se incorpora un nuevo símbolo proposicional al conjunto de inferencia, el número de modelos posibles se duplica (se multiplica por un factor de $2$). Esto se comprueba directamente en los resultados de la trayectoria: al avanzar de la posición $(1,1)$ con $n = 6$ símbolos hacia $(1,2)$ con $n = 10$ símbolos —incorporando $4$ símbolos nuevos—, el total de modelos posibles se multiplica por $2^4 = 16$, pasando de $64$ a $1,024$ asignaciones posibles.   

2. ¿Por qué el número de modelos crece exponencialmente?

    **Respuesta**

    El crecimiento es exponencial porque cada símbolo proposicional representa una variable binaria independiente que puede tomar dos valores de verdad (`True` o `False`). Al generar todas las combinaciones posibles de valores de verdad, cada nuevo símbolo divide en dos ramas cada una de las asignaciones existentes en el nivel previo del árbol de modelos, lo que da lugar a una progresión geométrica de base $2$ descrita por la función $M(n) = 2^n$.

3. ¿Qué implicaciones tiene este crecimiento para la utilización del algoritmo en tableros de mayor tamaño?

    **Respuesta**

    Este comportamiento exponencial vuelve computacionalmente intratable la verificación exhaustiva mediante tablas de verdad conforme aumenta la dimensión del entorno. Mientras que en el tablero actual de $4 \times 4$ existen $16$ casillas (lo que representa hasta $32$ símbolos solo para pozos y Troll, es decir, $2^{32} \approx 4.29 \times 10^9$ modelos posibles), escalar a un tablero de $10 \times 10$ ($100$ casillas y $200$ símbolos de peligro) generaría hasta $2^{200}$ combinaciones. Evaluar esa cantidad de modelos requeriría tiempos de cómputo inviables en la práctica, haciendo necesario recurrir a técnicas de inferencia más eficientes como resolución proposicional o encadenamiento hacia adelante y hacia atrás.

4. ¿Por qué es conveniente incluir únicamente los símbolos necesarios para evaluar las inferencias?

    **Respuesta**
    
    Porque cada símbolo adicional que se agrega al conjunto de evaluación duplica innecesariamente la profundidad y el tamaño del espacio de búsqueda sin aportar información útil sobre la casilla que se desea analizar. Al limitar el conjunto `simbolos` exclusivamente a las proposiciones relacionadas con las percepciones locales obtenidas hasta ese instante, se mantiene el valor de $n$ lo más reducido posible y se evita explorar asignaciones redundantes sobre regiones del tablero acerca de las cuales el agente aún no tiene conocimiento.   

In [27]:
def verificar_trayectoria(trayectoria):

    BC = lambda m: True
    simbolos = set()
    registros = []

    # Conocimiento inicial: la casilla (1,1) es segura
    BC = decir(BC, lambda m: not m['P_1_1'])
    BC = decir(BC, lambda m: not m['T_1_1'])

    simbolos.update(['P_1_1', 'T_1_1'])

    for paso in range(len(trayectoria) - 1):

        actual = trayectoria[paso]
        siguiente = trayectoria[paso + 1]

        i, j = actual

        brisa, hedor = percibir(actual)

        # Incorporar la percepción a la BC
        sentencia = regla_percepcion(i, j, brisa, hedor, N)
        BC = decir(BC, sentencia)

        # Actualizar símbolos
        simbolos = agregar_simbolos(simbolos, i, j, N)

        # Registrar métricas de complejidad en la posición actual
        num_simbolos = len(simbolos)
        modelos_posibles = 2 ** len(simbolos)
        registros.append((actual, num_simbolos, modelos_posibles))

        # Verificación del movimiento
        sig_i, sig_j = siguiente
        segura = es_segura(sig_i, sig_j, BC, simbolos)

        print("Posición actual:", actual)
        print("Percepción:", (brisa, hedor))
        print("Siguiente casilla:", siguiente)
        print("¿Es segura?:", segura)
        print()

        if not segura:
            print("Movimiento no autorizado.")
            break

    else:
        print("Trayectoria completada.")

    # Mostrar tabla con la información recopilada
    print("\n" + "=" * 54)
    print(f"{'Posición':<12} | {'Símbolos (n)':<15} | {'Modelos posibles (2^n)':<22}")
    print("-" * 54)
    for pos, n_simb, modelos in registros:
        print(f"{str(pos):<12} | {n_simb:<15} | {modelos:<22}")
    print("=" * 54)

In [28]:
verificar_trayectoria(trayectoria)

Posición actual: (1, 1)
Percepción: (False, False)
Siguiente casilla: (1, 2)
¿Es segura?: True

Posición actual: (1, 2)
Percepción: (False, False)
Siguiente casilla: (2, 2)
¿Es segura?: True

Posición actual: (2, 2)
Percepción: (True, False)
Siguiente casilla: (2, 3)
¿Es segura?: False

Movimiento no autorizado.

Posición     | Símbolos (n)    | Modelos posibles (2^n)
------------------------------------------------------
(1, 1)       | 6               | 64                    
(1, 2)       | 10              | 1024                  
(2, 2)       | 14              | 16384                 
